This one is about testing the DAE Only using normal FED AVG

In [ ]:
### Imports ###

import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import os
import numpy as np
import torch.nn as nn
import logging
from torch.optim import AdamW
import copy
import joblib
import time
import matplotlib.pyplot as plt
import json
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score, precision_score,
    recall_score, accuracy_score, matthews_corrcoef, balanced_accuracy_score,
    roc_curve, precision_recall_curve, confusion_matrix
)

In [ ]:
## Configurations & Hyperparameters ###
NUM_CLIENTS = 100
GLOBAL_ROUNDS = 300
CLIENT_EPOCHS_PER_ROUND = 15      
BATCH_SIZE = 64                   
LR = 1e-3
WEIGHT_DECAY = 1e-4
LATENT_DIM = 16
DROPOUT_P = 0.1
L1_LAMBDA = 1e-5
INITIAL_NOISE = 0.05
FINAL_NOISE = 0.02
PATIENCE = 10
INPUT_DIM = 36
CKKS_DOT_TIME = 0.14    
CKKS_ENC_TIME = 0.05    
CIPHERTEXT_SIZE_MB = 0.92 

# --- V2 Trust Parameters (Algorithm 1 & Section III-C) ---
TAU_SIM = 0.05         # Extremely lenient (almost 90-degree tolerance)
TAU_NORM_MULT = 10.0   # Massively widened so small clients don't fail magnitude
REWARD_ALPHA = 0.05    
PENALTY_BETA = 0.02   # Lowered: Prevents premature bannin

# Initialize Reliability Scores for all 20 nodes
reliability_scores = np.full(NUM_CLIENTS, 0.5)
ENCODER_PATH = r"/home/azwad/Works/IoMT_FL/Previous_Implementation/Dataset/after_scaling_encoding"
DATA_PATH = "/home/azwad/Works/IoMT_FL/Revised_Implementation/data/CIC_IoMT_2024"
OUTPUT_DIR = "/home/azwad/Works/IoMT_FL/Revised_Implementation/results/FL_Centralized_final_Attack_100"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(OUTPUT_DIR, exist_ok=True)


def setup_logging(log_file='training.log'):
    """Sets up a logger that writes to a file."""
    # Remove any existing handlers
    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)
    
    logger = logging.getLogger()
    logger.setLevel(logging.INFO)
    
    # Create file handler
    file_handler = logging.FileHandler(log_file, mode='w')
    file_handler.setLevel(logging.INFO)
    
    # Create formatter and add it to the handler
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    file_handler.setFormatter(formatter)
    
    # Add the handler to the logger
    logger.addHandler(file_handler)
    return logger

####  Setup & Logging ---

logger = setup_logging(log_file=os.path.join(OUTPUT_DIR, 'fl_train.log'))
logger.info(f"Starting FL Simulation (FedAvg) on {DEVICE}")
logger.info(f"Clients: {NUM_CLIENTS}, Rounds: {GLOBAL_ROUNDS}")


# ---------------- Adversarial Configuration ---------------- #

ATTACK_RATIO = 0.3   # 30% malicious clients
ATTACK_TYPES = ["sign_flip", "model_replacement", "backdoor", "collusion"]

NUM_ATTACKERS = int(NUM_CLIENTS * ATTACK_RATIO)

np.random.seed(42)
malicious_clients = np.random.choice(range(NUM_CLIENTS), NUM_ATTACKERS, replace=False)

client_attack_map = {}
for cid in malicious_clients:
    client_attack_map[cid] = np.random.choice(ATTACK_TYPES)

print("Malicious Clients:", client_attack_map)

In [ ]:
### Dataset Class & DataLoader Setup ###

class NIDSDataset(Dataset):

    def __init__(self, dataframe):

        self.X = dataframe.iloc[:, :-1].values

        self.y = dataframe.iloc[:, -1].values


        self.X = torch.tensor(self.X, dtype=torch.float32)
        self.y = torch.tensor(self.y, dtype=torch.int64)

    def __len__(self):

        return len(self.y)

    def __getitem__(self, idx):

        return self.X[idx], self.y[idx]

def get_dataloaders(base_path, batch_size):

    

    paths = {
        'train': os.path.join(base_path, 'train_benign.csv'),
        'val': os.path.join(base_path, 'val_benign.csv'),
        'test_balanced': os.path.join(base_path, 'test_balanced.csv'),
        'test_all': os.path.join(base_path, 'test_all.csv') 
    }
    

    try:
        dataframes = {
            name: pd.read_csv(path) for name, path in paths.items()
        }
    except FileNotFoundError as e:
        print(f"Error: Could not find data file at {e.filename}")
        print("Please check your DATA_PATH.")
        return None, None


    datasets = {
        name: NIDSDataset(df) for name, df in dataframes.items()
    }
    

    loaders = {
        'train': DataLoader(
            datasets['train'], 
            batch_size=batch_size, 
            shuffle=True, 
            num_workers=2, 
            pin_memory=True
        ),
        'val': DataLoader(
            datasets['val'], 
            batch_size=batch_size, 
            shuffle=False, 
            num_workers=2, 
            pin_memory=True
        ),
        'test_balanced': DataLoader(
            datasets['test_balanced'], 
            batch_size=batch_size, 
            shuffle=False, 
            num_workers=2, 
            pin_memory=True
        ),
        'test_all': DataLoader(
            datasets['test_all'], 
            batch_size=batch_size, 
            shuffle=False, 
            num_workers=2, 
            pin_memory=True
        )
    }


    input_dim = datasets['train'].X.shape[1]

    return loaders, input_dim




In [ ]:
### Partitioning for Federated Learning

def get_client_loaders(data_path, num_clients, batch_size):
    train_path = os.path.join(data_path, "train_benign.csv")
    val_path = os.path.join(data_path, "val_benign.csv") # For calculating val loss
    # We need a mixed validation set to calculate F1/AUROC during training rounds
    # We'll use the balanced test set for this 'monitoring' purpose strictly
    monitor_path = os.path.join(data_path, "test_balanced.csv") 
    
    
    train_df = pd.read_csv(train_path)
    val_df = pd.read_csv(val_path)
    monitor_df = pd.read_csv(monitor_path)
    
    # Shuffle and split training data
    train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
    client_dfs = np.array_split(train_df, num_clients)
    
    client_loaders = []
    for i, df in enumerate(client_dfs):
        ds = NIDSDataset(df)
        loader = torch.utils.data.DataLoader(
            ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True
        )
        client_loaders.append(loader)

    # Global loaders
    val_loader = torch.utils.data.DataLoader(
        NIDSDataset(val_df), batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True
    )
    # Monitoring loader (for F1/AUROC per round)
    monitor_loader = torch.utils.data.DataLoader(
        NIDSDataset(monitor_df), batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True
    )
    
    input_dim = client_loaders[0].dataset.X.shape[1]
    return client_loaders, val_loader, monitor_loader, input_dim

# --- 4. Load Label Encoder (Need Benign Label for Metrics) ---
try:
    label_encoder = joblib.load(os.path.join(ENCODER_PATH, 'label_encoder.joblib'))
    try:
        benign_label = int(label_encoder.transform(['Benign'])[0])
    except:
        benign_label = int(label_encoder.transform(['benign'])[0])
except Exception as e:
    logger.error(f"Could not load label encoder: {e}")
    exit()
#client_loaders, global_val_loader, global_monitor_loader, input_dim = get_client_loaders(
#    DATA_PATH, NUM_CLIENTS, BATCH_SIZE
#)
#print(f"Number of clients: {len(client_loaders)}")
#print(f"Batches per client: {len(client_loaders[0])}")
#print(f"Input dimension: {input_dim}")
#print(f"Global Val Loader batches: {len(global_val_loader)}")
#print(f"Global Monitor Loader batches: {len(global_monitor_loader)}")

In [ ]:
## Model Definition ###

class Encoder(nn.Module):

    def __init__(self, input_dim, latent_dim, dropout_p=0.1):
        super().__init__()
        
        # Main path
        self.main_path = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(dropout_p),
            nn.Linear(64, 32),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Dropout(dropout_p),
            nn.Linear(32, latent_dim)
        )
        
        
        self.skip_path = nn.Linear(input_dim, latent_dim)

    def forward(self, x):
        
        return self.main_path(x) + self.skip_path(x)

class Decoder(nn.Module):

    def __init__(self, latent_dim, output_dim, dropout_p=0.1):
        super().__init__()
        
        # Main path
        self.main_path = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Dropout(dropout_p),
            nn.Linear(32, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(dropout_p),
            nn.Linear(64, output_dim)
        )
        

        self.skip_path = nn.Linear(latent_dim, output_dim)

    def forward(self, z):

        return self.main_path(z) + self.skip_path(z)

class DAE(nn.Module):

    def __init__(self, input_dim, latent_dim, noise_factor=0.1, dropout_p=0.1):
        super().__init__()
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.noise_factor = noise_factor
        
        self.encoder = Encoder(
            input_dim=input_dim, 
            latent_dim=latent_dim, 
            dropout_p=dropout_p
        )
        
        self.decoder = Decoder(
            latent_dim=latent_dim, 
            output_dim=input_dim, 
            dropout_p=dropout_p
        )

    def forward(self, x):
        if self.training:
            noise = self.noise_factor * torch.randn_like(x)
            x_noisy = x + noise
        else:
            x_noisy = x
            

        z = self.encoder(x_noisy)
        

        x_recon = self.decoder(z)
        
        return x_recon



In [ ]:
# --- Engine ---

class EarlyStopping:
    """Implements early stopping with patience."""
    def __init__(self, patience=10, verbose=False, delta=0, path='best_model.pth', logger=None):
        self.patience = patience
        self.verbose = verbose
        self.delta = delta
        self.path = path
        self.logger = logger or logging.getLogger()
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                self.logger.info(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        if self.verbose:
            self.logger.info(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}). Saving model...')
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

def save_history(history, path='training_history.json'):
    """Saves the training history dictionary to a JSON file."""
    with open(path, 'w') as f:
        json.dump(history, f, indent=4)

def load_checkpoint(model, path='best_model.pth', device='cpu'):
    """Loads a model checkpoint from a file."""
    model.load_state_dict(torch.load(path, map_location=device))
    return model

def get_noise_factor(initial_noise, final_noise, epoch, total_epochs):
    """Anneals noise factor from initial to final (cosine schedule)."""
    
    cos_val = np.cos(np.pi * epoch / total_epochs)
    return final_noise + 0.5 * (initial_noise - final_noise) * (1 + cos_val)

def train_step(model, dataloader, criterion, optimizer, device, l1_lambda, current_noise_factor):
    """Performs a single training step (epoch)."""
    model.train()
    model.noise_factor = current_noise_factor # Set the noise factor
    
    total_recon_loss = 0.0
    total_l1_loss = 0.0
    
    for (features, _) in dataloader:
        features = features.to(device)
        
        optimizer.zero_grad()
        
        # --- DAE Forward Pass with L1 ---
        # 1. Add noise
        if model.training:
            noise = model.noise_factor * torch.randn_like(features)
            x_noisy = features + noise
        else:
            x_noisy = features # Should not happen in train_step, but safe
            
        # 2. Encode to get latent vector z
        z = model.encoder(x_noisy)
        
        # 3. Decode to get reconstruction
        x_recon = model.decoder(z)
        
        # --- Calculate Losses ---
        # 1. Reconstruction Loss (MSE)
        recon_loss = criterion(x_recon, features)
        
        # 2. L1 Sparsity Loss on latent vector
        l1_loss = l1_lambda * z.abs().mean()
        
        # 3. Total Loss
        loss = recon_loss + l1_loss
        
        loss.backward()
        optimizer.step()
        
        total_recon_loss += recon_loss.item()
        total_l1_loss += l1_loss.item()
        
    avg_recon_loss = total_recon_loss / len(dataloader)
    avg_l1_loss = total_l1_loss / len(dataloader)
    return avg_recon_loss, avg_l1_loss

def val_step(model, dataloader, criterion, device):
    """Performs a single validation step (epoch)."""
    model.eval()
    total_recon_loss = 0.0
    
    with torch.no_grad():
        for (features, _) in dataloader:
            features = features.to(device)
            
            # Forward pass (no noise, no L1)
            x_recon = model(features)
            
            # Reconstruction Loss
            recon_loss = criterion(x_recon, features)
            total_recon_loss += recon_loss.item()
            
    avg_recon_loss = total_recon_loss / len(dataloader)
    return avg_recon_loss

# --- Thresholding and Testing Logic ---

def find_threshold(model, dataloader, device, percentile=95):
    """
    Finds the reconstruction error threshold from the validation set.
    """
    model.eval()
    # Use MSELoss with 'none' to get per-sample errors
    criterion = nn.MSELoss(reduction='none')
    all_errors = []
    
    with torch.no_grad():
        for (features, _) in dataloader:
            features = features.to(device)
            x_recon = model(features)
            
            # Calculate per-sample error
            errors = criterion(x_recon, features).mean(dim=1)
            all_errors.append(errors.cpu().numpy())
            
    all_errors = np.concatenate(all_errors)
    
    # Set threshold at the nth percentile
    threshold = np.percentile(all_errors, percentile)
    return threshold

def test_model(model, dataloader, device, threshold, benign_label, label_encoder_classes):
    """
    Tests the model on the balanced test set and computes all metrics.
    """
    model.eval()
    criterion = nn.MSELoss(reduction='none')
    
    all_y_true = []
    all_y_true_binary = []
    all_y_pred_binary = []
    all_y_scores = []

    with torch.no_grad():
        for (features, labels) in dataloader:
            features = features.to(device)
            
            x_recon = model(features)
            
            # Per-sample reconstruction error (our anomaly score)
            errors = criterion(x_recon, features).mean(dim=1).cpu().numpy()
            
            # Predicted labels (1 = Attack, 0 = Benign)
            preds = (errors > threshold).astype(int)
            
            # True labels (binary)
            labels_np = labels.cpu().numpy()
            true_binary = (labels_np != benign_label).astype(int)
            
            all_y_true.extend(labels_np)
            all_y_true_binary.extend(true_binary)
            all_y_pred_binary.extend(preds)
            all_y_scores.extend(errors)

    # --- Calculate All Metrics ---
    results = {}
    
    y_true_b = np.array(all_y_true_binary)
    y_pred_b = np.array(all_y_pred_binary)
    y_scores = np.array(all_y_scores)
    y_true_multi = np.array(all_y_true)
    
    results['AUROC'] = roc_auc_score(y_true_b, y_scores)
    results['AUPRC'] = average_precision_score(y_true_b, y_scores)
    results['F1_Score'] = f1_score(y_true_b, y_pred_b)
    results['Precision'] = precision_score(y_true_b, y_pred_b)
    results['Recall'] = recall_score(y_true_b, y_pred_b)
    results['Accuracy'] = accuracy_score(y_true_b, y_pred_b)
    results['MCC'] = matthews_corrcoef(y_true_b, y_pred_b)
    results['Balanced_Accuracy'] = balanced_accuracy_score(y_true_b, y_pred_b)
    
    # TPR at 1% FPR
    fpr, tpr, _ = roc_curve(y_true_b, y_scores)
    results['TPR_at_1_FPR'] = np.interp(0.01, fpr, tpr)
    
    # FPR at 95% TPR
    # We need to sort by TPR to use interp
    sort_idx = np.argsort(tpr)
    tpr_sorted = tpr[sort_idx]
    fpr_sorted = fpr[sort_idx]
    results['FPR_at_95_TPR'] = np.interp(0.95, tpr_sorted, fpr_sorted)
    
    # Per-attack class recall
    per_class_recall = {}
    attack_labels = [i for i, cls in enumerate(label_encoder_classes) if cls not in ['Benign', 'benign']]
    
    for label_int in attack_labels:
        class_name = label_encoder_classes[label_int]
        
        # Get mask for samples of this specific attack class
        class_mask = (y_true_multi == label_int)
        
        if np.sum(class_mask) == 0:
            per_class_recall[class_name] = "N/A (No samples in test set)"
            continue
            
        class_y_true = y_true_b[class_mask]
        class_y_pred = y_pred_b[class_mask]
        
        # Calculate recall for this class (all should be '1')
        class_recall = recall_score(class_y_true, class_y_pred, zero_division=0)
        per_class_recall[class_name] = class_recall
        
    results['Per_Attack_Recall'] = per_class_recall

    return results

In [ ]:
# Initialize Global Model ---
global_model = DAE(
    input_dim=INPUT_DIM, 
    latent_dim=LATENT_DIM, 
    noise_factor=INITIAL_NOISE, 
    dropout_p=DROPOUT_P
).to(DEVICE)


In [ ]:
# Helper Functions ---

def fed_avg(weights_list, sample_counts):
    total_samples = sum(sample_counts)
    global_weights = copy.deepcopy(weights_list[0])
    for key in global_weights.keys():
        global_weights[key] = torch.zeros_like(global_weights[key], dtype=torch.float32)
        
    for weights, count in zip(weights_list, sample_counts):
        weight_factor = count / total_samples
        for key in weights.keys():
            global_weights[key] += weights[key] * weight_factor
    return global_weights


def calculate_approval_ratio(global_model, client_loaders, prev_re_errors, device, epsilon=0.15):
    """
    Implements Local Validation & Approval Ratio (Eq. 6 & 7).
    Every client evaluates the global model on their local benign data.
    """
    votes = []
    current_errors = []
    
    global_model.eval()
    criterion = nn.MSELoss(reduction='none')
    
    # Simulate multi-core parallel client verification
    # In a real system, this happens locally at each Fog node [cite: 143]
    with torch.no_grad():
        for i, loader in enumerate(client_loaders):
            batch_errors = []
            for features, _ in loader:
                features = features.to(device)
                recon = global_model(features)
                error = criterion(recon, features).mean().item()
                batch_errors.append(error)
            
            avg_re = np.mean(batch_errors)
            current_errors.append(avg_re)
            
            # Equation 6: Approve if Error is stable or decreasing
            # Approve (1) if RE_t <= RE_{t-1} + epsilon [cite: 247]
            if avg_re <= (prev_re_errors[i] + epsilon):
                votes.append(1)
            else:
                votes.append(0)
                
    approval_ratio = sum(votes) / len(votes) # Eq. 7 [cite: 252]
    return approval_ratio, current_errors, votes

def determine_next_sink(round_idx, reliability_scores, num_clients):
    """
    Implements Deterministic Sink Rotation (Eq. 10). [cite: 282, 289]
    Rotation happens every 5 rounds or upon Sink revocation.
    """
    # Filter for high-trust nodes (Top 35% pool as per Eq. 10) [cite: 284]
    indexed_scores = list(enumerate(reliability_scores))
    indexed_scores.sort(key=lambda x: x[1], reverse=True)
    
    pool_size = max(1, int(0.35 * num_clients))
    trusted_pool = [x[0] for x in indexed_scores[:pool_size]]
    
    # Deterministic selection using round index as a seed
    next_sink = trusted_pool[round_idx % pool_size]
    return next_sink


def calculate_communication_cost(model):
    """Estimates model size in MB."""
    param_size = 0
    for param in model.parameters():
        param_size += param.nelement() * param.element_size()
    buffer_size = 0
    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()
    
    size_all_mb = (param_size + buffer_size) / 1024**2
    return size_all_mb

# Calculate Communication overhead
model_size_mb = calculate_communication_cost(global_model)
logger.info(f"Global Model Size: {model_size_mb:.4f} MB")
# Cost per round = (Broadcast + Upload) * Num_Clients
comm_cost_per_round = model_size_mb * 2 * NUM_CLIENTS 

best_model_path = os.path.join(OUTPUT_DIR, 'global_best_model.pth')
early_stopper = EarlyStopping(
    patience=PATIENCE, verbose=True, path=best_model_path, logger=logger
)
criterion = nn.MSELoss()

# History tracker with FL metrics
history = {
    'round': [], 
    'train_loss': [], 
    'val_loss': [], 
    'val_f1': [], 
    'val_auroc': [],
    'cumulative_comm_mb': []
}

def evaluate_global_metrics(model, dataloader, threshold, benign_label):
    """Quick evaluation for F1 and AUROC during training rounds."""
    model.eval()
    criterion = nn.MSELoss(reduction='none')
    all_preds = []
    all_labels = []
    all_scores = []
    
    with torch.no_grad():
        for features, labels in dataloader:
            features = features.to(DEVICE)
            x_recon = model(features)
            errors = criterion(x_recon, features).mean(dim=1).cpu().numpy()
            
            preds = (errors > threshold).astype(int)
            # Convert labels to binary (0=Benign, 1=Attack)
            binary_labels = (labels.cpu().numpy() != benign_label).astype(int)
            
            all_preds.extend(preds)
            all_labels.extend(binary_labels)
            all_scores.extend(errors)
            
    f1 = f1_score(all_labels, all_preds)
    auroc = roc_auc_score(all_labels, all_scores)
    return f1, auroc


def attack_sign_flip(delta, scale=5.0):
    return {k: -scale * v for k, v in delta.items()}

def attack_model_replacement(local_w, global_w, gamma=10):
    attacked = {}
    for k in local_w.keys():
        attacked[k] = global_w[k] + gamma * (local_w[k] - global_w[k])
    return attacked

def attack_backdoor(delta, noise_std=0.05):
    attacked = {}
    for k, v in delta.items():
        attacked[k] = v + torch.randn_like(v) * noise_std
    return attacked

COLLUSION_VECTOR = None

def attack_collusion(delta):
    global COLLUSION_VECTOR
    if COLLUSION_VECTOR is None:
        COLLUSION_VECTOR = {k: torch.randn_like(v) for k, v in delta.items()}
    return {k: delta[k] + 3.0 * COLLUSION_VECTOR[k] for k in delta}

def attack_malicious_sink(global_weights):
    """
    Proposed Attack: Malicious Sink Hijack (Round 1).
    The Sink replaces the global model with extreme noise to destroy logic. [cite: 250, 251]
    """
    logger.warning("!!! ATTACK: Malicious Sink Hijack in Round 1 !!!")
    poisoned = {k: v + torch.randn_like(v) * 10.0 for k, v in global_weights.items()}
    return poisoned



def apply_attack(client_id, local_weights, global_weights):
    if client_id not in client_attack_map:
        return local_weights  # benign

    attack = client_attack_map[client_id]

    delta = {k: local_weights[k] - global_weights[k] for k in local_weights}

    if attack == "sign_flip":
        delta = attack_sign_flip(delta)

    elif attack == "model_replacement":
        return attack_model_replacement(local_weights, global_weights)

    elif attack == "backdoor":
        delta = attack_backdoor(delta)

    elif attack == "collusion":
        delta = attack_collusion(delta)

    attacked_weights = {k: global_weights[k] + delta[k] for k in delta}
    return attacked_weights

def trust_aware_aggregation(local_updates, delta_ref_flat, scores, round_idx):
    """
    Implements Secure Verification & Aggregation (Algorithm 1, Step 3).
    Checks magnitude and alignment of encrypted updates.
    """
    n_ref = torch.norm(delta_ref_flat).item()
    u_ref = delta_ref_flat / (n_ref + 1e-9)
    
    aggregated_delta = torch.zeros_like(delta_ref_flat)
    count_included = 0
    updated_scores = copy.deepcopy(scores)
    
    # Simulate CKKS parallel verification latency
    time.sleep(CKKS_DOT_TIME)

    for i, update in enumerate(local_updates):
        u_k = update['u']
        n_k = update['n']
        
        # 1. Alignment Check: Cosine Similarity (Equation 4) [cite: 198]
        # We use plaintext dot product to simulate CKKS decrypted result
        cos_sim = torch.dot(u_k, u_ref).item()
        
        # 2. Magnitude Check: Scalar Norm (Section III-C) [cite: 199]
        norm_check = n_k < (TAU_NORM_MULT * n_ref)
        
        # --- Reliability Scoring Update (Equation 5) [cite: 262] ---
        if cos_sim > TAU_SIM and norm_check:
            # Reward Factor alpha [cite: 265]
            updated_scores[i] = min(1.0, updated_scores[i] + REWARD_ALPHA)
            # Scalar Reconstruction and addition to aggregate 
            aggregated_delta += (u_k * n_k)
            count_included += 1
        else:
            # Penalty Factor beta [cite: 265]
            updated_scores[i] = max(0.0, updated_scores[i] - PENALTY_BETA)
            
    # Average the aggregated delta by the number of reliable contributors
    if count_included > 0:
        aggregated_delta /= count_included
        
    return aggregated_delta, updated_scores, count_included


def reconstruct_state_dict(base_model, flat_delta):
    """
    Reconstructs the model's state dictionary from a flattened update vector.
    This corresponds to the final Global Update step in the methodology. 
    """
    new_weights = {}
    curr_pos = 0
    # Iterate through the layers of the DAE model [cite: 170]
    for k, v in base_model.state_dict().items():
        numel = v.numel()
        # Slice the flat vector to get the weights for the current layer
        delta_segment = flat_delta[curr_pos:curr_pos + numel].view(v.shape)
        # Apply the update to the base weights 
        new_weights[k] = v + delta_segment
        curr_pos += numel
    return new_weights

In [ ]:
# --- 7. FL Training Loop (Ablation V2: Trust-Aware + Sensitivity Analysis) ---

client_loaders, global_val_loader, global_monitor_loader, input_dim = get_client_loaders(
    DATA_PATH, NUM_CLIENTS, BATCH_SIZE
)
# --- V3 Trust Constants (Section III-D & E) ---
# --- V3 Constants (Section III-D & E) ---
EPSILON_TOLERANCE = 0.15 # [cite: 245]
ALPHA_SINK = 0.05        # Reward for good sink [cite: 250]
BETA_SINK = 0.30         # Severe penalty for poisoning [cite: 250]
ROTATION_INTERVAL = 5    # regular rotation period [cite: 282]

# Verified thresholds from V2
t_sim = 0.5
n_mult = 2.5

logger.info("\n--- Starting Ablation V3: Full Robust System (Sink Rotation + Voting) ---")
start_time = time.time()

# trackers for plotting reliability and participation
reliability_history = [] # To track per-client scores over rounds
included_clients_history = []
attack_log = []

# 1. Step 0: System Initialization
global_model = DAE(input_dim=INPUT_DIM, latent_dim=LATENT_DIM, dropout_p=DROPOUT_P).to(DEVICE)
reliability_scores = np.full(NUM_CLIENTS, 0.5) # Neutral start [cite: 183, 206]
early_stopper = EarlyStopping(patience=PATIENCE, verbose=True, path=best_model_path, logger=logger)

# Establish initial baseline for Equation 6 [cite: 247]
initial_errors = []
global_model.eval()
with torch.no_grad():
    for i, loader in enumerate(client_loaders):
        batch_errors = []
        for features, _ in loader:
            features = features.to(DEVICE)
            recon = global_model(features)
            batch_errors.append(nn.MSELoss()(recon, features).mean().item())
        initial_errors.append(np.mean(batch_errors))
client_prev_errors = np.array(initial_errors) 

current_sink_id = determine_next_sink(0, reliability_scores, NUM_CLIENTS) 

for round_idx in range(GLOBAL_ROUNDS):
    current_noise = get_noise_factor(INITIAL_NOISE, FINAL_NOISE, round_idx, GLOBAL_ROUNDS)
    local_updates = []
    local_losses = []
    
    # --- 1. Sink Preparation (Algorithm 1, Step 1) ---
    sink_loader = client_loaders[current_sink_id]
    sink_model = copy.deepcopy(global_model)
    optimizer_sink = AdamW(sink_model.parameters(), lr=LR)
    train_step(sink_model, sink_loader, criterion, optimizer_sink, DEVICE, L1_LAMBDA, current_noise)
    
    delta_ref = {k: sink_model.state_dict()[k] - global_model.state_dict()[k] for k in sink_model.state_dict()}
    delta_ref_flat = torch.cat([v.flatten() for v in delta_ref.values()])
    n_ref = torch.norm(delta_ref_flat).item()
    u_ref = delta_ref_flat / (n_ref + 1e-9)

    # --- 2. Client Phase: Training & Decomposition (Algorithm 1, Step 2) ---
    for client_idx, loader in enumerate(client_loaders):
        local_model = copy.deepcopy(global_model)
        optimizer = AdamW(local_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        r_loss, l1_loss = train_step(local_model, loader, criterion, optimizer, DEVICE, L1_LAMBDA, current_noise)
        local_losses.append(r_loss + l1_loss)
        
        delta_k_dict = {k: local_model.state_dict()[k] - global_model.state_dict()[k] for k in local_model.state_dict()}
        delta_k_dict = apply_attack(client_idx, delta_k_dict, global_model.state_dict())
        
        flat_delta_k = torch.cat([v.flatten() for v in delta_k_dict.values()])
        n_k = torch.norm(flat_delta_k).item()
        u_k = flat_delta_k / (n_k + 1e-9)
        
        time.sleep(CKKS_ENC_TIME) 
        local_updates.append({'u': u_k, 'n': n_k, 'client_idx': client_idx})

    # --- 3. Server Phase: Secure Verification & Aggregation (Algorithm 1, Step 3) ---
    if round_idx == 9: # Round 10 Malicious Sink Attack [cite: 250, 251]
        new_global_weights = attack_malicious_sink(global_model.state_dict())
        count_included = 0
        logger.warning(f"SINK ATTACK: Malicious Sink Hijack by Node {current_sink_id}")
    else:
        time.sleep(CKKS_DOT_TIME) 
        aggregated_delta_flat = torch.zeros_like(delta_ref_flat)
        count_included = 0
        
        for update in local_updates:
            idx = update['client_idx']
            # Bidirectional Verification Check 
            if update['n'] < (n_mult * n_ref) and torch.dot(update['u'], u_ref).item() > t_sim:
                reliability_scores[idx] = min(1.0, reliability_scores[idx] + REWARD_ALPHA)
                aggregated_delta_flat += (update['u'] * update['n'])
                count_included += 1
            else:
                reliability_scores[idx] = max(0.0, reliability_scores[idx] - PENALTY_BETA)
        
        if count_included > 0: aggregated_delta_flat /= count_included
        new_global_weights = reconstruct_state_dict(global_model, aggregated_delta_flat)
    
    # --- 4. Sink Voting (Algorithm 1, Step 5) ---
    temp_model = copy.deepcopy(global_model)
    temp_model.load_state_dict(new_global_weights)
    approval_ratio, current_errors, votes = calculate_approval_ratio(
        temp_model, client_loaders, client_prev_errors, DEVICE, EPSILON_TOLERANCE
    )
    
    # Update Sink Reliability [cite: 252]
    if approval_ratio >= 0.6:
        reliability_scores[current_sink_id] = min(1.0, reliability_scores[current_sink_id] + ALPHA_SINK)
        global_model.load_state_dict(new_global_weights) # Accept
        client_prev_errors = np.array(current_errors)
        logger.info(f"Rd {round_idx+1} | Update ACCEPTED (Ratio: {approval_ratio:.2f})")
    else:
        reliability_scores[current_sink_id] = max(0.0, reliability_scores[current_sink_id] - BETA_SINK)
        logger.error(f"Rd {round_idx+1} | SINK REVOKED (Ratio: {approval_ratio:.2f})")

    # --- 5. Deterministic Sink Rotation (Section III-E) ---
    # Automatic every 5 rounds OR upon revocation [cite: 240, 282]
    if (round_idx + 1) % ROTATION_INTERVAL == 0 or approval_ratio < 0.6:
        current_sink_id = determine_next_sink(round_idx, reliability_scores, NUM_CLIENTS)
        logger.info(f"Rotation Policy: Node {current_sink_id} is now Sink")

    # --- 6. Metric Tracking & Logging ---
    val_recon_loss = val_step(global_model, global_val_loader, criterion, DEVICE)
    val_f1, val_auroc = evaluate_global_metrics(global_model, global_monitor_loader, PREVIOUS_OPTIMAL_THRESHOLD, benign_label)
    
    round_comm = (CIPHERTEXT_SIZE_MB * NUM_CLIENTS * 2) # Broadcast + Upload
    total_comm = round_comm * (round_idx + 1)
    
    # Store history
    history['round'].append(round_idx + 1)
    history['train_loss'].append(np.mean(local_losses))
    history['val_loss'].append(val_recon_loss)
    history['val_f1'].append(val_f1)
    history['val_auroc'].append(val_auroc)
    history['cumulative_comm_mb'].append(total_comm)
    reliability_history.append(copy.deepcopy(reliability_scores))
    included_clients_history.append(count_included)
    
    logger.info(f"Rd {round_idx+1} | Loss: {val_recon_loss:.6f} | F1: {val_f1:.4f} | "
                f"AUROC: {val_auroc:.4f} | Included: {count_included} | AvgRel: {np.mean(reliability_scores):.2f}")

    attack_log.append({"round": round_idx + 1, "val_loss": val_recon_loss, "f1": val_f1})
    
    early_stopper(val_recon_loss, global_model)
    if early_stopper.early_stop: break

total_training_time = time.time() - start_time
logger.info(f"--- Ablation V3 Complete in {total_training_time:.2f}s ---")

In [ ]:
# --- 8. Final Evaluation & Results Visualization ---
logger.info("\n--- Starting Final Evaluation (Proposed Framework) ---")

# Plot 1: Detection Robustness (F1 and Loss)
rounds = [x["round"] for x in attack_log]
f1_scores = [x["f1"] for x in attack_log]
losses = [x["val_loss"] for x in attack_log]

fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(rounds, f1_scores, color='green', label='F1 Score', marker='s', markevery=10)
ax1.set_xlabel('FL Rounds')
ax1.set_ylabel('F1 Score', color='green')
ax2 = ax1.twinx()
ax2.plot(rounds, losses, color='red', label='Recon Loss', linestyle='--')
ax2.set_ylabel('Validation Loss', color='red')
plt.title('Robustness Under Malicious Sink & Poisoning Attacks')
plt.grid(True, alpha=0.3)
plt.show()

# Plot 2: Reliability Score Dynamics (Eq. 5 / Fig 4)
rel_history_np = np.array(reliability_history)
plt.figure(figsize=(8, 5))
for cid in range(NUM_CLIENTS):
    label = "Attacker" if cid in malicious_clients else "Honest"
    color = "red" if cid in malicious_clients else "blue"
    alpha = 0.6 if cid in malicious_clients else 0.2
    plt.plot(rounds, rel_history_np[:, cid], color=color, alpha=alpha)

plt.title('Variation of Reliability Scores (Proposed System)')
plt.xlabel('Rounds')
plt.ylabel('Reliability Score $R_k^{(t)}$')
plt.legend(['Malicious Client (Banned)', 'Honest Client (High Trust)'], loc='lower right')
plt.show()

# Plot 3: Client Participation (Filter Performance)
plt.figure(figsize=(8, 4))
plt.bar(rounds, included_clients_history, color='teal')
plt.title('Number of Included Clients per Round (Filtering Malicious Updates)')
plt.xlabel('Rounds')
plt.ylabel('Included Count')
plt.show()

# Final Metrics Summary
results = test_model(global_model, global_monitor_loader, DEVICE, PREVIOUS_OPTIMAL_THRESHOLD, benign_label, label_encoder.classes_)
results['Total_Comm_MB'] = history['cumulative_comm_mb'][-1]
results['Final_Avg_Reliability'] = np.mean(reliability_scores)

logger.info("\n--- FINAL TEST RESULTS (V3) ---")
for k, v in results.items():
    if k != 'Per_Attack_Recall': logger.info(f"{k:<25}: {v}")